# 01 — Spark on ObjectScale
**Objective:** Spark 4.2 reads and writes ObjectScale S3 via the S3A connector.

**Customer sees:** a partitioned dataset written to OBS, listed in the OBS UI, read back and queried.
**TC:** TC1.1–TC1.6

## Step 1 — Session
Build SparkSession and print versions (masked creds).

In [ ]:
import pandas as pd
from IPython.display import HTML, display
from poc import config, results, viz, s3util
import poc.spark_helpers as sh
spark = sh.build_session("obs-poc")
viz.table(__import__("pandas").DataFrame(sh.versions(spark).items(), columns=["component","version"]))
print("users:", config.masked_users())
print("Spark UI:", config.SPARK_UI)

**Presenter notes** (hidden)

Same session config as the smoke test.

## Step 2 — Generate the dataset (TC1.1)
Synthetic sales orders (~200k rows, 30 days) generated in Spark — no external data needed.

In [ ]:
from poc import demo_data
df = demo_data.orders_df(spark)
res = results.write("TC1.1", "dataset generated", df.count() > 0,
                    {"rows": df.count()})
df.show(5); viz.banner(res["passed"], "TC1.1", f"{res['metrics']['rows']:,} rows")

**Presenter notes** (hidden)

Deterministic seed — same data every run.

## Step 3 — Write Parquet partitioned by date (TC1.2)
Data lands in `poc-spark-data/<run_id>/raw/orders` — one partition per order_date. Uses the S3A magic committer for atomic writes.

In [ ]:
import time, pandas as pd
path = demo_data.data_path()
t0 = time.time()
df.write.mode("overwrite").partitionBy("order_date").parquet(path)
res = results.write("TC1.2", "write partitioned parquet", True,
                    {"path": path, "seconds": round(time.time() - t0, 1)})
viz.banner(res["passed"], "TC1.2", f"{res['metrics']['seconds']}s -> {path}")

**Presenter notes** (hidden)

Show the Spark UI storage tab / stages during the write.

## Step 4 — Objects on ObjectScale (TC1.3)
List the objects Spark just wrote, then **pause and show the bucket in the ObjectScale UI** — same files, visible in the storage console.

In [ ]:
objs, _ = demo_data.list_objects(config.BUCKET_DATA, f"{results.run_id()}/raw/")
viz.object_table([(o["Key"], o["Size"]) for o in objs])
res = results.write("TC1.3", "objects listed in bucket", len(objs) > 0,
                    {"objects": len(objs)})
viz.banner(res["passed"], "TC1.3", f"{len(objs)} objects — show OBS UI now")

**Presenter notes** (hidden)

UI: https://192.168.1.31 — bucket poc-spark-data, prefix = run_id.

## Step 5 — Read back + business queries (TC1.4)
Read the Parquet dataset back and run three business questions as tables/charts: revenue by day, top customers, orders by status.

In [ ]:
import matplotlib.pyplot as plt, pandas as pd
rdf = spark.read.parquet(path).createOrReplaceTempView("orders")
q1 = spark.sql("SELECT order_date, round(sum(amount*qty),2) rev FROM orders GROUP BY 1 ORDER BY 1").toPandas()
q2 = spark.sql("SELECT customer, round(sum(amount*qty),2) rev FROM orders GROUP BY 1 ORDER BY rev DESC LIMIT 10").toPandas()
q3 = spark.sql("SELECT status, count(*) n FROM orders GROUP BY 1").toPandas()
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
q1.plot.bar(x="order_date", y="rev", ax=ax[0], legend=False); ax[0].set_title("revenue/day")
q2.plot.bar(x="customer", y="rev", ax=ax[1], legend=False); ax[1].set_title("top customers")
res = results.write("TC1.4", "read+queries", len(q1) > 0 and len(q2) > 0,
                    {"days": len(q1), "top": q2.iloc[0].customer})
viz.table(q3); plt.show(); viz.banner(res["passed"], "TC1.4")

**Presenter notes** (hidden)

Queries hit OBS via S3A on every executor.

## Step 6 — Integrity check (TC1.5)
Count parity between source and stored dataset proves no data loss.

In [ ]:
src, dst = df.count(), spark.read.parquet(path).count()
res = results.write("TC1.5", "integrity", src == dst, {"src": src, "dst": dst})
viz.banner(res["passed"], "TC1.5", f"{src:,} == {dst:,}")

**Presenter notes** (hidden)

Parquet column stats + partition pruning also verified implicitly.

## Step 7 — Multipart throughput vs concurrency (TC1.6)
Same 256 MiB object uploaded at 1, 2, 4, 8 and 16 parallel part streams — shows how concurrency scales sustained write throughput.

In [ ]:
from poc import throughput
rows = throughput.multipart_sweep(
    config.BUCKET_THROTTLE, f"{results.run_id()}/multipart",
    levels=(1, 2, 4, 8, 16), total_mb=256, part_mb=16)
sweep = pd.DataFrame(rows); viz.table(sweep)
res = results.write("TC1.6", "multipart write concurrency sweep", True,
                    {"levels": rows, "peak_mbs": max(r["mbs"] for r in rows)})
viz.banner(res["passed"], "TC1.6",
           f"peak {res['metrics']['peak_mbs']:.0f} MB/s")

**Presenter notes** (hidden)

Objects land under poc-throttle/<run_id>/multipart/ — cleaned by 99_reset.

## Step 8 — Throughput chart
Throughput vs concurrency from `reports/raw/TC1.6.json` — the ingest-scaling story.

In [ ]:
import pandas as pd, matplotlib.pyplot as plt
sw = pd.DataFrame(res["metrics"]["levels"]).set_index("concurrency")
sw.plot.line(y="mbs", marker="o", figsize=(6, 3.5), legend=False)
plt.xlabel("concurrent part streams"); plt.ylabel("MB/s")
plt.title("S3 multipart write throughput vs concurrency"); plt.grid(alpha=.3)
plt.show()
spark.stop()

**Presenter notes** (hidden)

Scaling tapers when a host/NIC limit is hit — expected on a 2-core lab VM.